# ✈️ Aviation Crisis Management & Financial Simulation (2015)

**Objective:** To simulate the financial impact of flight cancellations and diversions using Big Data (5.8M records), transforming operational anomalies into business insights.

In [1]:
# Importing Libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

- Loading & Merging Data

In [ ]:
# Loading 

flights_df=pd.read_csv(r'Airlines_db\flights.csv')
airlines_df=pd.read_csv(r'Airlines_db\airlines.csv')
airports_df=pd.read_csv(r'Airlines_db\airports.csv')

print(f'Flight shape : {flights_df.shape}\nAirlines shape : {airlines_df.shape}\nAirports shape: {airports_df.shape}')

## ⚙️ Phase 1: Data Ingestion & Memory Optimization
Merging datasets to replace ID codes with actual names, and immediately dropping redundant columns to save RAM.

In [ ]:
df_merged.info()

In [ ]:
num_cols = df_merged.select_dtypes(include= 'number').columns
for col in num_cols:

    print(col)
    print(df_merged[col].nunique())
    print(df_merged[col].unique())
    print('-' * 100)

In [ ]:
cat_cols = df_merged.select_dtypes(include= 'object').columns
for col in cat_cols:

    print(col)
    print(df_merged[col].nunique())
    print(df_merged[col].unique())
    print('-' * 100)

In [ ]:
# 1-Merging  flights with Airlines
df_merged = pd.merge(flights_df, airlines_df, left_on='AIRLINE', right_on='IATA_CODE', how='left')
df_merged.rename(columns={'AIRLINE_x':'AIRLINE_CODE','AIRLINE_y':'AIRLINE_NAME'},inplace=True)
df_merged.drop('IATA_CODE', axis=1, inplace=True)# as it repeated
df_merged.head()

In [ ]:
# 2- Merging the df_meged with the origin airports
airports_subset=airports_df[['IATA_CODE','AIRPORT', 'CITY']]
df_merged=pd.merge(df_merged,airports_subset,left_on='ORIGIN_AIRPORT',right_on='IATA_CODE',how='left')
df_merged.drop('IATA_CODE', axis=1, inplace=True)
df_merged.head()

In [ ]:
# 3- Merging df_meged with destination airports
df_merged = pd.merge(df_merged, airports_subset, left_on='DESTINATION_AIRPORT', right_on='IATA_CODE', how='left')
df_merged.rename(columns={'AIRPORT': 'DEST_AIRPORT_NAME', 'CITY': 'DEST_CITY'}, inplace=True)
df_merged.drop('IATA_CODE', axis=1, inplace=True)

In [ ]:
# As a result of merging, we have columns with the same name but different information, so we will rename them for better understanding and to avoid confusion.
df_merged.rename(columns={
    'AIRPORT_x': 'ORIGIN_AIRPORT_NAME',
    'CITY_x': 'ORIGIN_CITY',
    'AIRPORT_y': 'DEST_AIRPORT_NAME',
    'CITY_y': 'DEST_CITY'
}, inplace=True)
print(" Columns after renaming:")
print(df_merged.columns.tolist())

## 🧹 Phase 2: Defensive Data Cleaning & Business Logic
Handling Missing Values (NaNs) logically:
- Successful flights with no cancellation reason are explicitly marked as 'N'.
- Cancelled flights physically have 0 minutes of delay, air time, and taxi time.

notes to be applied : 
- as the data only in 2015 and the year only have one unique value so we have to delete it to save memory
- we observe that we had only 14 airlines for the 5.8M flights over 2015
- from the unique values of tail numbers we can observe that we had only 4,857 plane
- we had 322 for takeoff and 322 for landing distributed into 308 country



In [ ]:
(df_merged.isna().sum() / len(df_merged) * 100).round(2).sort_values(ascending=False)

In [ ]:
# 1. Dropping Redundant & Zero-Variance Columns
# WHY WE ARE DROPPING THESE COLUMNS:
# - 'YEAR': Dropped because it contains only a single unique value (2015). It has zero variance, provides no statistical value, and consumes unnecessary memory.
# - 'AIRLINE_CODE', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT': Their sole purpose was acting as merging keys to join our dimension tables. 
#    Since we successfully extracted the actual text names (AIRLINE_NAME, ORIGIN_AIRPORT_NAME, DEST_AIRPORT_NAME), these codes are now fully redundant.
# - 'TAIL_NUMBER', 'FLIGHT_NUMBER': Dropped because highly granular IDs are not needed for our macro-level economic simulation.

cols_to_drop=['YEAR',"AIRLINE_CODE",'ORIGIN_AIRPORT','DESTINATION_AIRPORT', 'TAIL_NUMBER', 'FLIGHT_NUMBER']
df_merged.drop(columns=cols_to_drop,inplace=True)

In [ ]:
# WHY WE ARE DOING THIS:
# 1. Consistency: The original column uses single-letter codes ('A', 'B', 'C', 'D'). 
#    Using a single letter like 'N' maintains categorical consistency and uses less memory than long strings.
# 2. Defensive Programming: We must ensure we only assign 'N' to flights that actually flew (CANCELLED == 0).
#    Any remaining missing values belong to cancelled flights with missing data entry, which we flag as 'U'.

# Step 1: Assign 'N' (Not Cancelled) to flights that successfully departed (CANCELLED == 0) but have a NaN reason.
# This prevents data corruption by explicitly categorizing successful flights based on strict logic.
df_merged['CANCELLATION_REASON'] = np.where(
    (df_merged['CANCELLED']==0) & (df_merged['CANCELLATION_REASON'].isnull()),
    'N',
    df_merged['CANCELLATION_REASON']
)
# Step 2: Assign 'U' (Unknown) to any remaining NaNs.
# Since successful flights are now marked as 'N', any remaining NaNs are guaranteed to be cancelled flights (CANCELLED == 1) 
# where the specific cancellation reason was not recorded by the airline. 'U' safely isolates these anomalies.
df_merged['CANCELLATION_REASON'] = df_merged['CANCELLATION_REASON'].fillna('U')


In [ ]:
# Handling Operational Timing & Durations
#------------------------------------------
# Why WE DOING THIS:
# These columns track physical events (e.g., TAXI_IN, AIR_TIME, WHEELS_ON). 
# The ~1.5% NaNs here directly correspond to the cancelled flights. Since a cancelled flight never leaves the gate, 
# its air time and taxi time are physically 0 minutes. 
# Filling these with 0 is crucial to prevent our future financial equations (e.g., Taxi Cost = TAXI_IN * $100) 
# from breaking or returning NaN errors.
Operational_columns=['ARRIVAL_DELAY', 'AIR_TIME', 'ELAPSED_TIME', 'WHEELS_ON', 
                       'ARRIVAL_TIME', 'TAXI_IN', 'TAXI_OUT', 'WHEELS_OFF', 
                       'DEPARTURE_DELAY', 'DEPARTURE_TIME']
df_merged[Operational_columns]=df_merged[Operational_columns].fillna(0)

In [ ]:
# As these columns mean that the plane not delay so we will put 0 instead of null
delay_columns = ['AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY', 'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY']
df_merged[delay_columns]=df_merged[delay_columns].fillna(0)

## 🕵️ Phase 3: The October Anomaly Fix
Addressing the 8.35% unmapped airports caused by an FAA system ID change in October.

In [ ]:
# investigating when the unkown Airports occurred
df_merged['ORIGIN_AIRPORT_NAME']=df_merged['ORIGIN_AIRPORT_NAME'].apply(lambda x: 'Unknown' if pd.isna(x) else x)
unkonwn_flights=df_merged[df_merged['ORIGIN_AIRPORT_NAME']=='Unknown']

missing_by_month=unkonwn_flights.groupby('MONTH').size().reset_index(name='Missing_Airports_Count')

fig_october = px.bar(missing_by_month, x='MONTH', y='Missing_Airports_Count', 
                     title='Anomaly Detection: Missing Airport Names by Month (2015)',
                     labels={'MONTH': 'Month of the Year', 'Missing_Airports_Count': 'Number of Unmapped Airports'},
                     text='Missing_Airports_Count',
                     color='Missing_Airports_Count', color_continuous_scale='Reds')

fig_october.update_xaxes(tickmode='linear', dtick=1)
fig_october.update_traces(textposition='outside')
fig_october.show()
# WHY WE ARE DOING THIS:
# The 8.35% missing values correspond to the flights in October where the FAA switched from IATA letters to numeric IDs.
# Since we dropped the redundant code columns for memory optimization, we will label these as 'Unknown' 
# to keep the dataset clean and ready for visualization without dropping valuable rows.


In [ ]:
unmapped_cols = {
    'ORIGIN_CITY': 'Unknown',
    'DEST_AIRPORT_NAME': 'Unknown',
    'DEST_CITY': 'Unknown'
}

for col, fill_val in unmapped_cols.items():
    df_merged[col] = df_merged[col].fillna(fill_val)

In [ ]:
# saving the dataframe to a CSV file
df_merged.to_csv('Cleaned_Airlines_Crisis.csv', index=False)

In [2]:
Cleaned_df=pd.read_csv('Cleaned_Airlines_Crisis.csv')
Cleaned_df.head()

,MONTH,DAY,DAY_OF_WEEK,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,TAXI_OUT,WHEELS_OFF,SCHEDULED_TIME,ELAPSED_TIME,...,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY,AIRLINE_NAME,ORIGIN_AIRPORT_NAME,ORIGIN_CITY,DEST_AIRPORT_NAME,DEST_CITY
0,1,1,4,5,2354.0,-11.0,21.0,15.0,205.0,194.0,...,0.0,0.0,0.0,0.0,0.0,Alaska Airlines Inc.,Ted Stevens Anchorage International Airport,Anchorage,Seattle-Tacoma International Airport,Seattle
1,1,1,4,10,2.0,-8.0,12.0,14.0,280.0,279.0,...,0.0,0.0,0.0,0.0,0.0,American Airlines Inc.,Los Angeles International Airport,Los Angeles,Palm Beach International Airport,West Palm Beach
2,1,1,4,20,18.0,-2.0,16.0,34.0,286.0,293.0,...,0.0,0.0,0.0,0.0,0.0,US Airways Inc.,San Francisco International Airport,San Francisco,Charlotte Douglas International Airport,Charlotte
3,1,1,4,20,15.0,-5.0,15.0,30.0,285.0,281.0,...,0.0,0.0,0.0,0.0,0.0,American Airlines Inc.,Los Angeles International Airport,Los Angeles,Miami International Airport,Miami
4,1,1,4,25,24.0,-1.0,11.0,35.0,235.0,215.0,...,0.0,0.0,0.0,0.0,0.0,Alaska Airlines Inc.,Seattle-Tacoma International Airport,Seattle,Ted Stevens Anchorage International Airport,Anchorage


## 🧠 Phase 4.2: Advanced Financial Engineering (Tier-Based Reputation Cost)
**Objective:** Move beyond static financial penalties by implementing a dynamic model based on airline business models and brand reputation.

**Business Logic & Assumptions:**
1. **Estimated Base Revenue:** Calculated as `Flight Distance * $25` (industry average for revenue per seat mile on a standard flight).
2. **Dynamic Cancellation Penalty:** Not all airlines compensate passengers equally.
   - **Premium Airlines (e.g., Delta, American):** High penalty (`$75,000`). They provide hotel vouchers, premium rebooking, and incur higher costs to protect their Brand Reputation.
   - **Budget Airlines (e.g., Spirit, Frontier):** Low penalty (`$25,000`). They typically offer mere refunds with minimal alternative accommodations.
   - **Standard Airlines:** Standard penalty (`$50,000`).
   
By calculating the **Total Penalty as a percentage of Total Estimated Revenue**, we reveal the *true* impact of the crisis on an airline's profit margin, not just the raw dollar amount lost.

In [3]:
# Calculate Estimated Base Revenue For successful flights
# Assumption: 25$ per mile flown. Cancelled flights yield 0$.
Cleaned_df['Estimated_Revenue']=np.where(
    Cleaned_df['CANCELLED']==0,
    Cleaned_df['DISTANCE']*25,
    0
)
# Define Airline Business Tiers (Comprehensive List)
premium_airlines = [
    'American Airlines Inc.', 'Delta Air Lines Inc.', 'United Air Lines Inc.', 
    'US Airways Inc.', 'Alaska Airlines Inc.', 'Hawaiian Airlines Inc.'
]

budget_airlines = [
    'Spirit Air Lines', 'Frontier Airlines Inc.'
]

# Note: Any cancelled flight by an airline NOT in the above two lists 
# (e.g., Southwest, JetBlue, Skywest) will automatically fall into the Standard penalty (Condition 3).

# Build Conditions for Dynamic Penalty
conditions=[
    (Cleaned_df['CANCELLED']==1)&(Cleaned_df['AIRLINE_NAME'].isin(premium_airlines)),
    (Cleaned_df['CANCELLED']==1)&(Cleaned_df['AIRLINE_NAME'].isin(budget_airlines)),
    (Cleaned_df['CANCELLED']==1)# Standard/Regional Airlines catch_all
]
# Define Pentlty Value for each condition
choices=[
    75000, #Permium penalty
    25000, # Budget pentaly
    50000
]

Cleaned_df['Cancellation_Penalty']=np.select(conditions,choices,default=0)


In [4]:
# Revenue vs. Reputation-Based Penalties

# Aggregate data per airline
airline_finance=Cleaned_df.groupby('AIRLINE_NAME').agg(
    Total_Revenue=('Estimated_Revenue','sum'),
    Total_Penality=("Cancellation_Penalty",'sum')
).reset_index()

# Calculate Penalty as a percentage of Total Revenue
airline_finance['loss_percentage']=(airline_finance['Total_Penality'] / airline_finance['Total_Revenue']).round(2) * 100

# Sort airlines from worst to best
airline_finance= airline_finance.sort_values(by='loss_percentage',ascending=False)

# Plot the results
fig_reputation = px.bar(
    airline_finance, 
    x='loss_percentage', 
    y='AIRLINE_NAME', 
    orientation='h',
    title='<b>The Real Impact: Which Airlines Suffered the Most?</b><br><sup>(Cancellation Penalty as a % of Total Estimated Revenue)</sup>',
    labels={'loss_percentage': '% of Revenue Lost to Cancellations', 'AIRLINE_NAME': 'Airline'},
    color='loss_percentage', 
    color_continuous_scale='Reds'
)

fig_reputation.update_layout(yaxis={'categoryorder':'total ascending'})
fig_reputation.show()

## 📈 Phase 5.2: Airline Financial Health (Revenue vs. Bleed)
**Objective:** Compare the total operational revenue of airlines against their financial bleed from cancellations.

To provide a complete financial picture, we visualize:
1. **The Winners (Revenue):** Top 10 airlines generating the most revenue from successful flights.
2. **The Bleeders (Losses):** Top 10 airlines losing the most money due to dynamic cancellation penalties.

In [20]:
airline_finance.head(10)

,AIRLINE_NAME,Total_Revenue,Total_Penality,loss_percentage
2,American Eagle Airlines Inc.,2957435975,751250000,25.0
3,Atlantic Southeast Airlines,6432107950,761550000,12.0
11,US Airways Inc.,4459841400,305025000,7.0
8,Skywest Airlines Inc.,7201113325,498000000,7.0
1,American Airlines Inc.,18642482400,818925000,4.0
9,Southwest Airlines Co.,23113083400,802150000,3.0
12,United Air Lines Inc.,16192560600,492975000,3.0
7,JetBlue Airways,6992717450,213800000,3.0
4,Delta Air Lines Inc.,18615653200,286800000,2.0
10,Spirit Air Lines,2843740700,50100000,2.0


In [17]:
# Chart 1:
top_Revenue=airline_finance.sort_values(by='Total_Revenue',ascending=False).head(10)
fig_rev=px.bar(
    top_Revenue,
    x='Total_Revenue',
    y='AIRLINE_NAME',
    orientation='h',
    title='<b>Top 10 Airlines by Estimated Revenue (Successful Flights)</b>',
    labels={'Total_Revenue':'Total Revenue (USD)','AIRLINE_NAME':'Airline'},
    color_continuous_scale='greens',
    color='Total_Revenue'
)
fig_rev.update_layout(yaxis={'categoryorder':'total ascending'})
fig_rev.show()

In [19]:
top_losses = airline_finance.sort_values(by='Total_Penality', ascending=False).head(10)

fig_loss = px.bar(
    top_losses, 
    x='Total_Penality', 
    y='AIRLINE_NAME', 
    orientation='h',
    title='<b>Top 10 Airlines by Financial Bleed (Cancellation Penalties)</b>',
    labels={'Total_Penality': 'Total Penalty (USD)', 'AIRLINE_NAME': 'Airline'},
    color='Total_Penality', 
    color_continuous_scale='Reds' # Red signifies financial loss
)

# Sorting the bars so the largest is at the top
fig_loss.update_layout(yaxis={'categoryorder':'total ascending'})
fig_loss.show()

In [ ]:
Cleaned_df['AIRLINE_NAME'].unique()

## 📊 Phase 5: The Executive Dashboard
Visualizing the overall financial impact, top bleeding airlines, and top profiting airports.

### Phase 2: Outlier Detection & Business Justification

- WHY WE ARE NOT DROPPING OUTLIERS:
- In standard predictive modeling, outliers are removed to normalize the distribution.
- However, in 'Crisis Management & Financial Simulation', the outliers ARE the crisis!
- Extreme delays (e.g., 1000+ minutes) represent real-world black swan events (severe weather, system failures) 
- that cause the highest financial bleed. Dropping them would invalidate our crisis simulation.

In [ ]:
df_sample = Cleaned_df.sample(frac=0.3, random_state=42)

fig_outliers = px.box(df_sample, x='AIRLINE_NAME', y='DEPARTURE_DELAY',
                      title='Outlier Analysis: The Reality of Extreme Delays (Black Swan Events)',
                      labels={'DEPARTURE_DELAY': 'Departure Delay (Minutes)', 'AIRLINE_NAME': 'Airline'},
                      color='AIRLINE_NAME')
fig_outliers.update_layout(showlegend=False)
fig_outliers.show()
print("🚨 The Top 5 Most Extreme Delay Outliers (The Crises):")
extreme_delays = df_merged[['AIRLINE_NAME', 'ORIGIN_CITY', 'DEST_CITY', 'DEPARTURE_DELAY', 'WEATHER_DELAY']].sort_values(by='DEPARTURE_DELAY', ascending=False).head(10)
display(extreme_delays)